# Modelo 2.2 - CrossValidation por atacante (LOAO)

Este cuaderno implementa una validacion cruzada **Leave-One-Attack-Out (LOAO)** entre atacantes para evaluar robustez de generalizacion en deteccion de spoofing.

Modelos evaluados:
- `A_spatial_cnn`
- `C_independent_frames`

Idea experimental:
1. Unir `train + dev`.
2. En cada iteracion, dejar fuera un `attack_id` de spoof para test.
3. Entrenar con el resto de `attack_id` de spoof.
4. Mantener una particion fija de bonafide para train/test, evitando leakage.
5. Repetir para todos los atacantes y comparar estabilidad por modelo.

## Justificacion metodologica

La validacion LOAO mide una pregunta de negocio clave: **que ocurre cuando aparece un atacante no visto durante entrenamiento**.

- Si un modelo funciona solo con atacantes conocidos, su valor operativo es limitado.
- Si mantiene rendimiento al dejar fuera un `attack_id`, su robustez es mayor.
- Ayuda a determinar qué atacante es más difícil de detectar.

Se priorizan metricas robustas en desbalance (`balanced_accuracy`, `MCC`, `F1_spoof`, `ROC-AUC`, `PR-AUC`) y se complementan con analisis grafico por atacante.

In [1]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
    matthews_corrcoef,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option('display.max_columns', 200)
plt.style.use('default')

In [2]:
def ensure_4d(x):
    x = np.asarray(x)
    if x.ndim == 4:
        return x
    if x.ndim == 3:
        return np.expand_dims(x, axis=-1)
    if x.ndim == 5 and x.shape[-1] == 1:
        return np.squeeze(x, axis=-1)
    raise ValueError(f'Shape no soportado para Conv2D: {x.shape}')

def compile_binary_model(model, lr=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
    )
    return model

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1, zero_division=0
    )
    out = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_spoof': precision,
        'recall_spoof': recall,
        'f1_spoof': f1,
        'roc_auc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'log_loss': log_loss(y_true, np.clip(y_prob, 1e-7, 1 - 1e-7)),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
    }
    return out

def robust_predict_proba(model, x):
    try:
        return model.predict(x, batch_size=64, verbose=0).ravel()
    except UnboundLocalError as e:
        if 'batch_outputs' not in str(e):
            raise
        return model(x, training=False).numpy().ravel()

In [3]:
def build_option_a_spatial_cnn(input_shape):
    model = models.Sequential(name='A_spatial_cnn')
    model.add(layers.Input(shape=input_shape))
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((1, 8), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((1, 8), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    return compile_binary_model(model)

def build_option_c_independent_frames(input_shape):
    inp = layers.Input(shape=input_shape, name='input_4d')
    x = layers.TimeDistributed(layers.Conv1D(32, 9, padding='same', activation='relu'))(inp)
    x = layers.TimeDistributed(layers.MaxPooling1D(4))(x)
    x = layers.TimeDistributed(layers.Conv1D(64, 9, padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.MaxPooling1D(4))(x)
    x = layers.TimeDistributed(layers.Conv1D(128, 5, padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.GlobalMaxPooling1D())(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=inp, outputs=out, name='C_independent_frames')
    return compile_binary_model(model)

MODEL_BUILDERS = {
    'A_spatial_cnn': build_option_a_spatial_cnn,
    'C_independent_frames': build_option_c_independent_frames,
}

## Carga de datos desde `npy` + protocols oficiales

En esta version:
- `y_labels.npy` aporta solo etiqueta binaria (`y`: 0 bonafide, 1 spoof),
- `attack_id` se toma desde los protocolos CM (`train/dev/eval`).

Luego se alinea cada split para mantener consistencia entre `X`, `y` y `attack_id`.

In [4]:
RUTA_TRAIN = '../Metricas/ETL_V2.1_train/'
RUTA_DEV = '../Metricas/ETL_V2.1_dev/'
RUTA_EVAL = '../Metricas/ETL_V2.1_eval/'

PROTOCOL_TRAIN = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
PROTOCOL_DEV = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
PROTOCOL_EVAL = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'


def _to_binary_label(v):
    if isinstance(v, (np.integer, int, np.floating, float, bool)):
        return int(v)
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {'0', 'bonafide', 'bona', 'real', 'human'}:
            return 0
        if s in {'1', 'spoof', 'fake', 'sintetico', 'synthetic'}:
            return 1
    raise ValueError(f'No se pudo convertir etiqueta a binario: {v}')


def load_cm_protocol(protocol_path):
    columns = ['speaker_id', 'file_name', 'system_id', 'attack_id', 'label']
    df = pd.read_csv(protocol_path, sep=' ', names=columns)
    out = df[['file_name', 'attack_id', 'label']].copy()
    out['file_name'] = out['file_name'].astype(str)
    out['attack_id'] = out['attack_id'].astype(str)
    out['y'] = out['label'].map(_to_binary_label).astype('int32')
    return out


def align_protocol_to_y(protocol_df, y_raw, split_name):
    y_arr = np.array([_to_binary_label(v) for v in np.asarray(y_raw)], dtype='int32')
    p_labels = protocol_df['y'].to_numpy(dtype='int32')

    if len(y_arr) > len(protocol_df):
        raise ValueError(
            f'{split_name}: y tiene mas filas que el protocolo '
            f'({len(y_arr)} > {len(protocol_df)}).'
        )

    # Alineacion por subsecuencia: robusta cuando ETL omite filas fallidas sin reordenar.
    chosen = []
    j = 0
    for i, yv in enumerate(y_arr):
        while j < len(p_labels) and p_labels[j] != yv:
            j += 1
        if j == len(p_labels):
            raise ValueError(
                f'{split_name}: no se logro alinear y[{i}]={yv} contra protocolo. '
                'Revisa orden o archivos faltantes en ETL.'
            )
        chosen.append(j)
        j += 1

    meta = protocol_df.iloc[chosen].copy().reset_index(drop=True)
    meta['split'] = split_name
    meta['y'] = y_arr

    return meta, y_arr


# Carga de tensores
X_train = ensure_4d(np.load(os.path.join(RUTA_TRAIN, 'X_fourier_features.npy'))).astype('float32')
y_train_raw = np.load(os.path.join(RUTA_TRAIN, 'y_labels.npy'), allow_pickle=True)
X_dev = ensure_4d(np.load(os.path.join(RUTA_DEV, 'X_fourier_features.npy'))).astype('float32')
y_dev_raw = np.load(os.path.join(RUTA_DEV, 'y_labels.npy'), allow_pickle=True)

y_eval_raw = np.load(os.path.join(RUTA_EVAL, 'y_labels.npy'), allow_pickle=True)

# Carga de protocolos train/dev/eval
protocol_train = load_cm_protocol(PROTOCOL_TRAIN)
protocol_dev = load_cm_protocol(PROTOCOL_DEV)
protocol_eval = load_cm_protocol(PROTOCOL_EVAL)

# Alineacion protocolo <-> y por split
meta_train, y_train = align_protocol_to_y(protocol_train, y_train_raw, 'train')
meta_dev, y_dev = align_protocol_to_y(protocol_dev, y_dev_raw, 'dev')
meta_eval, y_eval = align_protocol_to_y(protocol_eval, y_eval_raw, 'eval')

# Vectores de attack_id solicitados
attack_id_train = meta_train['attack_id'].to_numpy(dtype=str)
attack_id_dev = meta_dev['attack_id'].to_numpy(dtype=str)
attack_id_eval = meta_eval['attack_id'].to_numpy(dtype=str)

# Chequeos de consistencia
if len(meta_train) != len(X_train):
    raise ValueError(f'No coincide train: X={len(X_train)} vs y/meta={len(meta_train)}')
if len(meta_dev) != len(X_dev):
    raise ValueError(f'No coincide dev: X={len(X_dev)} vs y/meta={len(meta_dev)}')
if meta_train['attack_id'].isna().any() or meta_dev['attack_id'].isna().any() or meta_eval['attack_id'].isna().any():
    raise ValueError('Se detectaron attack_id nulos tras alinear protocolos.')

# LOAO se construye con train+dev (eval queda disponible para analisis externo)
X_all = np.concatenate([X_train, X_dev], axis=0)
meta_all = pd.concat([meta_train, meta_dev], ignore_index=True)
y_all = meta_all['y'].values.astype('int32')

if len(meta_all) != len(X_all):
    raise ValueError(f'No coincide combinado: X={len(X_all)} vs meta={len(meta_all)}')

print('Shape combinado train+dev:', X_all.shape, y_all.shape)
print('Attack IDs detectados (train+dev):', sorted(meta_all['attack_id'].astype(str).unique().tolist())[:15], '...')
print('Vectores attack_id | train/dev/eval:', len(attack_id_train), len(attack_id_dev), len(attack_id_eval))
print('Attack IDs unicos eval (muestra):', sorted(pd.Series(attack_id_eval).unique().tolist())[:15], '...')
display(meta_all.head())

ValueError: y_labels.npy no contiene attack_id en un formato soportado. Se esperaba estructura con attack_id y etiqueta por muestra.

In [ ]:
# Diagnostico robusto de labels/campos disponibles
# Funciona incluso si la celda anterior fallo al construir meta_all.

def inspect_y_raw(name, arr):
    print('\n' + '=' * 90)
    print(f'{name}')
    print('=' * 90)
    print('type:', type(arr))
    print('dtype:', getattr(arr, 'dtype', None))
    print('shape:', getattr(arr, 'shape', None))

    # Caso 1: ndarray estructurado
    if hasattr(arr, 'dtype') and arr.dtype.names is not None:
        print('Campos estructurados detectados:', arr.dtype.names)
        sample = pd.DataFrame(arr[:5])
        print('Muestra de registros:')
        display(sample)
        return

    arr_obj = np.array(arr, dtype=object)

    # Caso 2: matriz 2D
    if arr_obj.ndim == 2:
        print('Matriz 2D detectada. Primeras filas (hasta 5):')
        display(pd.DataFrame(arr_obj[:5]))
        return

    # Caso 3: vector de dicts
    if arr_obj.ndim == 1 and len(arr_obj) > 0 and isinstance(arr_obj[0], dict):
        keys = sorted({k for d in arr_obj[:50] for k in d.keys()})
        print('Lista de keys detectadas en dicts:', keys)
        print('Muestra de dicts:')
        display(pd.DataFrame(list(arr_obj[:5])))
        return

    # Caso 4: vector simple (probablemente solo 0/1)
    if arr_obj.ndim == 1:
        s = pd.Series(arr_obj)
        print('Vector 1D detectado (posible etiqueta binaria simple).')
        print('Valores unicos (top 20):')
        print(s.astype(str).value_counts().head(20))
        print('Primeros 20 valores:')
        print(s.head(20).tolist())
        return

    print('Formato no reconocido automaticamente.')


if 'y_train_raw' in globals():
    inspect_y_raw('y_train_raw', y_train_raw)
else:
    print('y_train_raw no esta definido. Ejecuta primero la celda de carga.')

if 'y_dev_raw' in globals():
    inspect_y_raw('y_dev_raw', y_dev_raw)
else:
    print('y_dev_raw no esta definido. Ejecuta primero la celda de carga.')

# Si meta_all existe (cuando la carga haya funcionado), tambien mostrar resumen final.
if 'meta_all' in globals():
    print('\nColumnas disponibles en meta_all:')
    print(list(meta_all.columns))
    print('\nTipos de datos por columna:')
    print(meta_all.dtypes)
    print('\nMuestra de meta_all:')
    display(meta_all.head(10))


y_train_raw
type: <class 'numpy.ndarray'>
dtype: int64
shape: (25380,)
Vector 1D detectado (posible etiqueta binaria simple).
Valores unicos (top 20):
1    22800
0     2580
Name: count, dtype: int64
Primeros 20 valores:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

y_dev_raw
type: <class 'numpy.ndarray'>
dtype: int64
shape: (24844,)
Vector 1D detectado (posible etiqueta binaria simple).
Valores unicos (top 20):
1    22296
0     2548
Name: count, dtype: int64
Primeros 20 valores:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Definicion de folds LOAO por `attack_id`

- Folds sobre spoof: cada iteracion deja fuera un `attack_id` de spoof completo.
- Bonafide: se usa una particion fija train/test (estratificada por semilla) para mantener clase negativa en ambos conjuntos sin mezclar muestras.

In [ ]:
spoof_mask = meta_all['y'].values == 1
bonafide_mask = meta_all['y'].values == 0

attack_values = meta_all.loc[spoof_mask, 'attack_id'].astype(str)
attack_ids = sorted(attack_values.unique().tolist())

idx_bonafide = np.where(bonafide_mask)[0]
idx_bona_train, idx_bona_test = train_test_split(
    idx_bonafide,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

print('Numero de attack_id spoof:', len(attack_ids))
print('Attack IDs:', attack_ids)
print('Bonafide train/test:', len(idx_bona_train), len(idx_bona_test))

In [ ]:
EPOCHS = 30
BATCH_SIZE = 32

callbacks = [
    EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=0,
    )
]

rows = []
input_shape = X_all.shape[1:]

for holdout_attack in attack_ids:
    idx_spoof_test = np.where((meta_all['attack_id'].astype(str).values == holdout_attack) & spoof_mask)[0]
    idx_spoof_train = np.where((meta_all['attack_id'].astype(str).values != holdout_attack) & spoof_mask)[0]

    idx_train = np.concatenate([idx_spoof_train, idx_bona_train])
    idx_test = np.concatenate([idx_spoof_test, idx_bona_test])

    X_train_fold, y_train_fold = X_all[idx_train], y_all[idx_train]
    X_test_fold, y_test_fold = X_all[idx_test], y_all[idx_test]

    classes = np.unique(y_train_fold)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_fold)
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

    print('\n' + '=' * 90)
    print(f'LOAO fold | attack_id test = {holdout_attack} | train={len(idx_train)} | test={len(idx_test)}')
    print('=' * 90)

    for model_name, builder in MODEL_BUILDERS.items():
        model = builder(input_shape)
        t0 = time.time()
        model.fit(
            X_train_fold,
            y_train_fold,
            validation_split=0.15,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=0,
        )
        train_time = time.time() - t0

        y_prob = robust_predict_proba(model, X_test_fold)
        fold_metrics = compute_metrics(y_test_fold, y_prob, threshold=0.5)
        fold_metrics['model'] = model_name
        fold_metrics['holdout_attack_id'] = holdout_attack
        fold_metrics['n_train'] = int(len(idx_train))
        fold_metrics['n_test'] = int(len(idx_test))
        fold_metrics['train_time_sec'] = float(train_time)
        rows.append(fold_metrics)

results_loao = pd.DataFrame(rows)
display(results_loao.head())

In [ ]:
metrics_core = ['balanced_accuracy', 'f1_spoof', 'roc_auc', 'pr_auc', 'mcc', 'log_loss']

summary_mean = (
    results_loao.groupby('model')[metrics_core + ['train_time_sec']]
    .mean()
    .sort_values('balanced_accuracy', ascending=False)
)

summary_std = results_loao.groupby('model')[metrics_core].std()

print('=== Media de metricas por modelo (LOAO attack_id) ===')
display(summary_mean.round(4))

print('=== Desviacion estandar de metricas por modelo ===')
display(summary_std.round(4))

print('=== Ranking por fold (balanced_accuracy) ===')
display(
    results_loao[['holdout_attack_id', 'model', 'balanced_accuracy', 'f1_spoof', 'roc_auc', 'mcc']]
    .sort_values(['holdout_attack_id', 'balanced_accuracy'], ascending=[True, False])
    .reset_index(drop=True)
    .round(4)
)

In [ ]:
pivot_bacc = results_loao.pivot(index='holdout_attack_id', columns='model', values='balanced_accuracy')
pivot_mcc = results_loao.pivot(index='holdout_attack_id', columns='model', values='mcc')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im1 = axes[0].imshow(pivot_bacc.values, aspect='auto', cmap='YlGnBu')
axes[0].set_title('Heatmap Balanced Accuracy por attack_id')
axes[0].set_xticks(range(len(pivot_bacc.columns)))
axes[0].set_xticklabels(pivot_bacc.columns, rotation=20)
axes[0].set_yticks(range(len(pivot_bacc.index)))
axes[0].set_yticklabels(pivot_bacc.index)
fig.colorbar(im1, ax=axes[0], fraction=0.03)

im2 = axes[1].imshow(pivot_mcc.values, aspect='auto', cmap='OrRd')
axes[1].set_title('Heatmap MCC por attack_id')
axes[1].set_xticks(range(len(pivot_mcc.columns)))
axes[1].set_xticklabels(pivot_mcc.columns, rotation=20)
axes[1].set_yticks(range(len(pivot_mcc.index)))
axes[1].set_yticklabels(pivot_mcc.index)
fig.colorbar(im2, ax=axes[1], fraction=0.03)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['balanced_accuracy', 'f1_spoof', 'roc_auc']):
    results_loao.boxplot(column=metric, by='model', ax=ax)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.grid(alpha=0.3)
plt.suptitle('Distribucion de metricas por modelo en LOAO')
plt.tight_layout()
plt.show()

## Analisis e interpretacion academica

Guia para redactar resultados:

1. **Rendimiento promedio**: usar `summary_mean` para identificar que modelo domina en media.
2. **Estabilidad entre atacantes**: usar `summary_std` y los boxplots para medir variabilidad fold a fold.
3. **Sensibilidad a atacantes concretos**: usar heatmaps para detectar `attack_id` especialmente dificiles.
4. **Trade-off operativo**: contrastar `balanced_accuracy` con `MCC` y errores (`FP/FN`) segun riesgo de negocio.

Hipotesis interpretativas sugeridas:
- Si `C_independent_frames` supera en media pero con alta varianza, podria ser mas potente in-domain pero sensible a shift de atacante.
- Si `A_spatial_cnn` mantiene menor varianza, podria estar capturando patrones mas transferibles entre atacantes.
- Atacantes con caidas simultaneas en ambos modelos sugieren necesidad de nuevas features o augmentation especifica.

Conclusiones esperadas:
- ranking global por robustez,
- ranking por estabilidad,
- y recomendaciones de siguiente experimento (calibracion por fold, ensamble, analisis de errores por atacante).